In [3]:
from fastaUtils import *

In [4]:
# Download the full Swiss-Prot database
swissprotFile = downloadDataBase(sourceURL="https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz",
                                 outputFile="swissprot.fasta")

  https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
  This may take several minutes...
Progress: 100.0%
Download complete. Decompressing...
  The database was downloaded successfully!
  File: swissprot.fasta


In [5]:
# run an analysis on the sequences in the downloaded database
swissProtStats = analyzeFastaFile("./swissprot.fasta")
# print the statistics
printFastaStats(swissProtStats)


Analysis for: swissprot.fasta
Number of sequences:   574,627
Total length (bp/aa):  208,482,574

Length statistics:
  Average length:      362.81
  Standard deviation:  344.74
  Minimum length:      2
  Maximum length:      35213

Quartiles:
  25%: 169.00
  50% (median): 295.00
  75%: 450.00


# MMseqs2 expriments

In [1]:
from mmseqs2Utils import MMseqsUtils

In [2]:
tool = MMseqsUtils()

In [3]:
queryDB = tool.createDB("queries.fasta", "./customDatabase/query", "queryDB")
targetDB = tool.createDB("swissprot.fasta", "./customDatabase/target", "targetDB")

In [4]:
resultsPath = "./customDatabase/results.tsv"
resultsDF = tool.runSearch(queryDB, targetDB, resultsPath, threads = 16)

Search completed with sensitivity = 7.5


In [5]:
print(resultsDF.head(n=10).to_string(index=False))

     query target  pident  alnlen  mismatch  gapopen  qstart  qend  tstart  tend       evalue  bits  qlen  tlen  qcov  tcov
A0A0F2LCD1 A4WKS9    52.4     114        53        0       5   117       8   121 3.520000e-29   115   122   121 0.926 0.942
A0A0F2LCD1 Q8ZV65    52.5     111        52        0       8   117      11   121 3.214000e-28   112   122   121 0.902 0.917
A0A0F2LCD1 A4YIV8    47.3      88        46        0       2    89      14   101 1.064000e-16    79   122   113 0.721 0.779
A0A0F2LCD1 Q97ZY6    47.6      81        42        0      10    90      22   102 8.797000e-15    74   122   120 0.664 0.675
A0A0F2LCD1 B6YW35    42.6      90        51        0       1    90       1    90 7.990000e-14    71   122    98 0.738 0.918
A0A0F2LCD1 Q5JIY8    42.2      90        51        0       1    90       1    90 1.500000e-13    70   122    98 0.738 0.918
A0A0F2LCD1 Q8U442    42.2      90        51        0       1    90       1    90 1.500000e-13    70   122    99 0.738 0.909
A0A0F2LC

In [ ]:
runtimes = []
numOfThreads = [1, 2, 4, 8, 16, 32]

for numThreads in numOfThreads:
    resultFileName = f"./customDatabase/results_{numThreads}threads.tsv"
    t0 = time.time()
    results = runMmseqsSearch(queryDB, targetDB, resultFileName, threads = numThreads)
    t1 = time.time()
    runtimes.append(t1-t0)
    print(f"Number of threads used: {numThreads}.")

print(runtimes)
np.savetxt('./customDatabase/runTime_threads.csv', runtimes, delimiter=',')

# Diamond experiments

In [1]:
from diamondUtils import DiamondUtils

In [2]:
tool = DiamondUtils()

In [3]:
queryDB = "./queries.fasta"
targetDB = tool.createDB("swissprot.fasta", "./customDatabase/target_diamond", "targetDB")

Preparing database...


In [5]:
resultsPath = "./customDatabase/results_diamond.tsv"
results = tool.runSearch(queryDB, targetDB, resultsPath, threads = 16, mode='ultra-sensitive')

Search completed in the mode: ultra-sensitive


In [6]:
print(results.head(n=10).to_string(index=False))

tr|W0FSK4|W0FSK4_9FLAV sp|Q6YMS3|POLG_DEN3M  99.6  1880   8  0  1  1880.1  1.1  1880.2  0.0   3792
tr|W0FSK4|W0FSK4_9FLAV sp|Q6YMS4|POLG_DEN3S  98.9  1880  20  0  1    1880    1    1880  0.0 3774.0
tr|W0FSK4|W0FSK4_9FLAV sp|Q5UB51|POLG_DEN3I  98.2  1880  33  0  1    1880    1    1880  0.0 3748.0
tr|W0FSK4|W0FSK4_9FLAV sp|Q99D35|POLG_DEN3C  98.1  1880  35  0  1    1880    1    1880  0.0 3743.0
tr|W0FSK4|W0FSK4_9FLAV sp|P27915|POLG_DEN3P  97.8  1880  41  0  1    1880    1    1880  0.0 3734.0
tr|W0FSK4|W0FSK4_9FLAV sp|P27909|POLG_DEN1B  75.5  1882 459  1  1    1880    1    1882  0.0 3043.0
tr|W0FSK4|W0FSK4_9FLAV sp|P17763|POLG_DEN1W  75.5  1882 460  1  1    1880    1    1882  0.0 3041.0
tr|W0FSK4|W0FSK4_9FLAV sp|P33478|POLG_DEN1S  75.3  1882 462  2  1    1880    1    1881  0.0 3028.0
tr|W0FSK4|W0FSK4_9FLAV sp|P07564|POLG_DEN2J  68.0  1882 600  2  1    1880    1    1881  0.0 2729.0
tr|W0FSK4|W0FSK4_9FLAV sp|P14340|POLG_DEN2N  67.7  1882 604  2  1    1880    1    1881  0.0 2721.0
tr|W0FSK4|

In [ ]:
runtimes = []
numOfThreads = [1, 2, 4, 8, 16, 32]

for numThreads in numOfThreads:
    resultFileName = f"./customDatabase/results_{numThreads}threads.tsv"
    t0 = time.time()
    results = runDiamondSearch(queryDB, targetDB, resultFileName, sensitivity='ultra-sensitive', threads = numThreads)
    t1 = time.time()
    runtimes.append(t1-t0)
    print(f"Number of threads used: {numThreads}.")

print(runtimes)
np.savetxt('./customDatabase/runTime_threads.csv', runtimes, delimiter=',')